#Research assignment on Machine Learning

Data ingestion from an oine source (github)

In [0]:
# Run this in a notebook cell only if the packages are not installed:
# %pip install pandas scikit-learn seaborn matplotlib

# Import pandas
import pandas as pd

# Download and load the Titanic dataset directly from GitHub
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"

df = pd.read_csv(url)

# Display the TABLE
display(df)



survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,3,male,22.0,1,0,7.25,S,Third,man,true,null,Southampton,no,false
1,1,female,38.0,1,0,71.2833,C,First,woman,false,C,Cherbourg,yes,false
1,3,female,26.0,0,0,7.925,S,Third,woman,false,null,Southampton,yes,true
1,1,female,35.0,1,0,53.1,S,First,woman,false,C,Southampton,yes,false
0,3,male,35.0,0,0,8.05,S,Third,man,true,null,Southampton,no,true
0,3,male,null,0,0,8.4583,Q,Third,man,true,null,Queenstown,no,true
0,1,male,54.0,0,0,51.8625,S,First,man,true,E,Southampton,no,true
0,3,male,2.0,3,1,21.075,S,Third,child,false,null,Southampton,no,false
1,3,female,27.0,0,2,11.1333,S,Third,woman,false,null,Southampton,yes,false
1,2,female,14.0,1,0,30.0708,C,Second,child,false,null,Cherbourg,yes,false


#Explore and clean the data

In [0]:
# Display the number of rows and columns
print("Dataset shape:", df.shape)



Dataset shape: (891, 15)


In [0]:
# Show column names, data types and non-missing values
df.info()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   survived     891 non-null    int64  
 1   pclass       891 non-null    int64  
 2   sex          891 non-null    object 
 3   age          714 non-null    float64
 4   sibsp        891 non-null    int64  
 5   parch        891 non-null    int64  
 6   fare         891 non-null    float64
 7   embarked     889 non-null    object 
 8   class        891 non-null    object 
 9   who          891 non-null    object 
 10  adult_male   891 non-null    bool   
 11  deck         203 non-null    object 
 12  embark_town  889 non-null    object 
 13  alive        891 non-null    object 
 14  alone        891 non-null    bool   
dtypes: bool(2), float64(2), int64(4), object(7)
memory usage: 92.4+ KB


In [0]:
# Display descriptive statistics for numeric columns
display(df.describe())



survived,pclass,age,sibsp,parch,fare
891.0,891.0,714.0,891.0,891.0,891.0
0.3838383838383838,2.308641975308642,29.69911764705882,0.5230078563411896,0.38159371492704824,32.204207968574636
0.4865924542648575,0.836071240977049,14.526497332334042,1.1027434322934317,0.8060572211299483,49.6934285971809
0.0,1.0,0.42,0.0,0.0,0.0
0.0,2.0,20.125,0.0,0.0,7.9104
0.0,3.0,28.0,0.0,0.0,14.4542
1.0,3.0,38.0,1.0,0.0,31.0
1.0,3.0,80.0,8.0,6.0,512.3292


In [0]:
# Count missing values in each column
print("\nMissing values:")
print(df.isnull().sum())



Missing values:
survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64


In [0]:

# Count duplicate rows
print( df.duplicated().sum())



107


In [0]:


# Display the data types of all columns
print("\nData types:")
print(df.dtypes)


Data types:
survived         int64
pclass           int64
sex             object
age            float64
sibsp            int64
parch            int64
fare           float64
embarked        object
class           object
who             object
adult_male        bool
deck            object
embark_town     object
alive           object
alone             bool
dtype: object


#Select the relevant columns and handle missing values
We do not need every Titanic column for this exercise. We will select our six columns and then inspect their missing values.

In [0]:
# Select only the columns needed for this assignment
df = df[
    ["age", "fare", "pclass", "sex", "embarked", "survived"]
].copy()

# Check missing values in the selected columns
print("Missing values before cleaning:")
print(df.isnull().sum())

# Check duplicate rows
print("Duplicate rows before cleaning:", df.duplicated().sum())

Missing values before cleaning:
age         177
fare          0
pclass        0
sex           0
embarked      2
survived      0
dtype: int64
Duplicate rows before cleaning: 120


# Duplicate investigation:
 120 rows were identified as duplicates across the selected columns.

 However, the dataset does not contain a unique PassengerId column.

 Therefore, we cannot confirm whether these records belong to
 the same passengers or different passengers.
 
 To avoid removing potentially valid observations, we will retain them.

#Separate the features from the target

In [0]:
# X contains the columns the model will use as inputs
X = df[["age", "fare", "pclass", "sex", "embarked"]]

# y contains the outcome we want to predict
y = df["survived"]

# Display the first five rows of the features
display(X.head())

# Display the first five target values
display(y.head())

age,fare,pclass,sex,embarked
22.0,7.25,3,male,S
38.0,71.2833,1,female,C
26.0,7.925,3,female,S
35.0,53.1,1,female,S
35.0,8.05,3,male,S


0    0
1    1
2    1
3    1
4    0
Name: survived, dtype: int64

- Features (X) are the information used to make a prediction.
- Target (y) is the answer the model is trying to predict.

We do not encode or scale survived as if it were an input feature. It remains the target for this classification task.

#Split the data into training and testing sets

In [0]:
# Import the function used to split the data
from sklearn.model_selection import train_test_split

# Split the data into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X,                  # Input features
    y,                  # Target values
    test_size=0.20,     # Reserve 20% for testing
    random_state=42,    # Make the split reproducible
    stratify=y          # Preserve survival class proportions approximately
)




In [0]:
# Import the function used to split the data
from sklearn.model_selection import train_test_split


# Display the size of each resulting dataset
print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)



Training features: (712, 5)
Testing features: (179, 5)


In [0]:


print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)



Training target: (712,)
Testing target: (179,)


In [0]:


# Check the proportion of each class in the two sets
print("\nTraining target proportions:")
print(y_train.value_counts(normalize=True))




Training target proportions:
survived
0    0.616573
1    0.383427
Name: proportion, dtype: float64


In [0]:


print("\nTesting target proportions:")
print(y_test.value_counts(normalize=True))


Testing target proportions:
survived
0    0.614525
1    0.385475
Name: proportion, dtype: float64


Understand these three important parameters:
- test_size=0.20 reserves 20% of the observations for testing.
- random_state=42 makes the random split repeatable when the same data and settings are used.
- stratify=y helps preserve the relative proportions of survivors and non-survivors in both sets.
The train_test_split function is documented in sklearn.model_selection. 
scikit-learn 1.9.1 documentation+1


Why split before preprocessing? Because the test data must represent data the training process has not used to learn preprocessing values.

#Handle missing values, encode categories and scale numeric columns

In [0]:
# Import tools for preprocessing
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
    MinMaxScaler
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline



In [0]:
# Identify the numeric input columns
numeric_features = ["age", "fare"]

# Identify the categorical input columns
categorical_features = ["pclass", "sex", "embarked"]

# Create a pipeline for numeric columns
numeric_pipeline = Pipeline(steps=[
    # Fill missing numeric values using the median
    ("imputer", SimpleImputer(strategy="median")),

    # Standardise the numeric values using the training mean and standard deviation
    ("scaler", StandardScaler())
])

# Create a pipeline for categorical columns
categorical_pipeline = Pipeline(steps=[
    # Fill missing categories with the most frequent category
    ("imputer", SimpleImputer(strategy="most_frequent")),

    # Convert categories into binary indicator columns
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])



In [0]:

# Apply the appropriate pipeline to each group of columns
preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

# Learn the preprocessing values from the training data and transform it
X_train_processed = preprocessor.fit_transform(X_train)

# Apply the already learned preprocessing to the test data
X_test_processed = preprocessor.transform(X_test)

# Display the dimensions of the processed datasets
print("Processed training data:", X_train_processed.shape)
print("Processed testing data:", X_test_processed.shape)



Processed training data: (712, 10)
Processed testing data: (179, 10)


In [0]:
# Confirm that no missing values remain in the processed feature matrices
import numpy as np

print(
    "Missing values in training features:",
    np.isnan(X_train_processed.data).sum()
    if hasattr(X_train_processed, "tocsr")
    else np.isnan(X_train_processed).sum()
)

print(
    "Missing values in testing features:",
    np.isnan(X_test_processed.data).sum()
    if hasattr(X_test_processed, "tocsr")
    else np.isnan(X_test_processed).sum()
)

Missing values in training features: 0
Missing values in testing features: 0


# Compare the original and processed data

In [0]:
# Retrieve the names of the columns created by preprocessing
feature_names = preprocessor.get_feature_names_out()

# Convert the processed training data to a DataFrame for easier viewing
# The toarray() call converts a sparse matrix into a dense array for display
X_train_preview = pd.DataFrame(
    X_train_processed.toarray()
    if hasattr(X_train_processed, "toarray")
    else X_train_processed,
    columns=feature_names,
    index=X_train.index
)

# Display the first five rows of the processed training data
display(X_train_preview.head())



numeric__age,numeric__fare,categorical__pclass_1,categorical__pclass_2,categorical__pclass_3,categorical__sex_female,categorical__sex_male,categorical__embarked_C,categorical__embarked_Q,categorical__embarked_S
-0.08113533197997112,0.5138115038918489,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0
-0.08113533197997112,-0.6625632269369692,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0
-0.08113533197997112,3.9553985809041556,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
-0.8878271899997933,-0.4678743521312413,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0
0.11093415802474844,-0.11597681371233177,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0


In [0]:
# Display the mean and standard deviation of the scaled numeric columns
print("Scaled numeric column means:")
print(X_train_preview[
    [f"numeric__{col}" for col in numeric_features]
].mean())

print("\nScaled numeric column standard deviations:")
print(X_train_preview[
    [f"numeric__{col}" for col in numeric_features]
].std())

Scaled numeric column means:
numeric__age     9.934703e-17
numeric__fare   -3.492836e-17
dtype: float64

Scaled numeric column standard deviations:
numeric__age     1.000703
numeric__fare    1.000703
dtype: float64


#Use Min-Max normalisation Example

In [0]:
# Import the MinMaxScaler
from sklearn.preprocessing import MinMaxScaler

# Create a small example dataset with two numeric features
sample_data = pd.DataFrame({
    "age": [20, 30, 40, 50, 60],
    "salary": [5000, 10000, 20000, 40000, 80000]
})

print("Original data:")
display(sample_data)

Original data:


age,salary
20,5000
30,10000
40,20000
50,40000
60,80000


In [0]:
# Create the scaler
minmax_scaler = MinMaxScaler()

# Fit the scaler and transform the sample data
sample_scaled = minmax_scaler.fit_transform(sample_data)

# Convert the result back into a readable DataFrame
sample_scaled_df = pd.DataFrame(
    sample_scaled,
    columns=sample_data.columns
)



display(sample_scaled_df)

age,salary
0.0,0.0
0.25,0.06666666666666667
0.5,0.2
0.75,0.4666666666666667
1.0,1.0


Min-max scaling transforms each feature using its observed minimum and maximum, ordinarily mapping the training values into the range 0 to 1. Standardisation instead centres features around their training means and scales them by their standard deviations. Neither method automatically makes outliers disappear.

#Final validation

In [0]:
# Confirm that training and testing rows do not overlap
print(
    "Overlapping row indices:",
    len(set(X_train.index).intersection(set(X_test.index)))
)

# Confirm that the training and test datasets have the same feature columns
print(
    "Same number of processed features:",
    X_train_processed.shape[1] == X_test_processed.shape[1]
)

# Confirm that the target remains separate from the processed features
print("Training target length:", len(y_train))
print("Testing target length:", len(y_test))

# Confirm that the split accounts for all rows in the selected dataset
print("Total split rows:", len(X_train) + len(X_test))
print("Original selected rows:", len(df))

Overlapping row indices: 0
Same number of processed features: True
Training target length: 712
Testing target length: 179
Total split rows: 891
Original selected rows: 891
